# 🏠 House Price Predictor - Model Training & Comparison

In this notebook, we'll train several machine learning models to predict house prices using the cleaned Ames Housing dataset. We'll compare **Ridge Regression**, **Random Forest**, and **XGBoost** to find the best performing model.

We'll also visualize model performances and feature importances.

In [ ]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from xgboost import XGBRegressor

from utils import PROCESSED_DATA_PATH, rmse, mae

# Set plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

## 1. Load Processed Data

In [ ]:
df = pd.read_csv(PROCESSED_DATA_PATH)
print(f"Dataset shape: {df.shape}")
df.head()

In [ ]:
# Split into features (X) and target (y)
X = df.drop(columns=["SalePrice"])
y = df["SalePrice"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train set: {X_train.shape[0]} rows")
print(f"Test set: {X_test.shape[0]} rows")

## 2. Model Training & Cross-Validation

We log-transform `SalePrice` (`np.log1p`) during training because housing prices are typically right-skewed. After predicting, we apply `np.expm1` to convert back to dollar amounts.

In [ ]:
models = {
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=10))
    ]),
    "Random Forest": RandomForestRegressor(
        n_estimators=200, max_depth=15, random_state=42, n_jobs=-1
    ),
    "XGBoost": XGBRegressor(
        n_estimators=500, learning_rate=0.05, max_depth=4,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbosity=0
    ),
}

def evaluate_model(model, name):
    # Train
    model.fit(X_train, np.log1p(y_train))
    
    # CV on log prices
    cv_scores = cross_val_score(
        model, X_train, np.log1p(y_train), 
        cv=5, scoring='neg_root_mean_squared_error'
    )
    cv_rmse = -cv_scores.mean()
    
    # Evaluaton on Test set
    preds_log = model.predict(X_test)
    preds = np.expm1(preds_log)
    test_rmse = rmse(y_test, preds)
    test_mae = mae(y_test, preds)
    test_r2 = model.score(X_test, np.log1p(y_test))
    
    return {
        "Name": name,
        "CV RMSE (log)": cv_rmse,
        "Test RMSE ($)": np.expm1(test_rmse),
        "Test MAE ($)": test_mae,
        "Test R2": test_r2,
        "Predictions": preds
    }

results = []
for name, model in models.items():
    print(f"Training {name}...")
    results.append(evaluate_model(model, name))

results_df = pd.DataFrame([{
    k: v for k, v in r.items() if k != "Predictions"
} for r in results])
results_df.set_index("Name").round(4)

## 3. Visualizing Results

In [ ]:
plt.figure(figsize=(10, 5))
sns.barplot(data=results_df, x="Name", y="Test R2", palette="viridis")
plt.title("Test R² Score by Model", fontsize=14)
plt.ylim(0.8, 1.0)
plt.ylabel("R² Score")
plt.show()

In [ ]:
best_result = min(results, key=lambda x: x["CV RMSE (log)"])
best_model_name = best_result["Name"]
print(f"🏆 Best Model: {best_model_name}")

plt.figure(figsize=(8, 8))
sns.scatterplot(x=y_test, y=best_result["Predictions"], alpha=0.6, color="#2a9d8f")

# Diagonal line for perfect translation
min_val = min(y_test.min(), best_result["Predictions"].min())
max_val = max(y_test.max(), best_result["Predictions"].max())
plt.plot([min_val, max_val], [min_val, max_val], color="#e76f51", linestyle="--")

plt.title(f"Actual vs Predicted Prices ({best_model_name})")
plt.xlabel("Actual SalePrice ($)")
plt.ylabel("Predicted SalePrice ($)")
plt.show()

## 4. Feature Importance

Let's look at what features the XGBoost model found most important.

In [ ]:
xgb_model = models["XGBoost"]
importances = pd.Series(xgb_model.feature_importances_, index=X.columns)
top_features = importances.sort_values(ascending=False).head(15)

plt.figure(figsize=(10, 6))
sns.barplot(x=top_features.values, y=top_features.index, palette="mako")
plt.title("Top 15 Feature Importances (XGBoost)")
plt.xlabel("Importance Score")
plt.show()

## 5. Conclusion

The model training here mirrors `src/train.py`, which actively trains the model and saves the best one (XGBoost) to `models/model.pkl`.
The analysis confirms that `TotalSF`, `Overall Qual`, and `Gr Liv Area` are the strongest drivers of house prices in Ames.